In [2]:
from dataclasses import dataclass


@dataclass
class RecipeRow:
    id: str
    name: str
    source_url: str
    source_name: str


@dataclass
class RecipeResult:
    id: str
    status: str
    final_url: str | None
    image_url: str | None
    image_path: str | None
    notes: str | None

In [3]:
import asyncio

import requests


HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 RecipeBot/1.0"
    )
}


def _fetch_html_sync(url: str) -> tuple[int, str]:
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=20,
        allow_redirects=True,
    )

    return response.status_code, response.text


async def fetch_html(url: str) -> tuple[int, str]:
    return await asyncio.to_thread(
        _fetch_html_sync,
        url,
    )

In [4]:
from bs4 import BeautifulSoup
import requests
from recipe_scrapers import scrape_html


HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 RecipeBot/1.0"
    )
}


def _extract_from_metadata(html: str) -> dict | None:
    soup = BeautifulSoup(html, "html.parser")

    title = None
    image = None

    og_title = soup.find("meta", property="og:title")
    if og_title and og_title.get("content"):
        title = og_title["content"].strip()

    og_image = soup.find("meta", property="og:image")
    if og_image and og_image.get("content"):
        image = og_image["content"].strip()

    if not title and soup.title and soup.title.string:
        title = soup.title.string.strip()

    if title and image:
        return {
            "title": title,
            "image": image,
        }

    return None


def extract_recipe(url: str) -> dict | None:
    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=20,
            allow_redirects=True,
        )
        response.raise_for_status()

        try:
            scraper = scrape_html(
                response.text,
                org_url=url,
            )

            return {
                "title": scraper.title(),
                "image": scraper.image(),
            }
        except Exception as scraper_error:
            print(f"Scraper parse failed for {url}: {scraper_error}")

        metadata_result = _extract_from_metadata(response.text)
        if metadata_result:
            return metadata_result

        print(f"Metadata fallback failed for {url}: missing title/image")
        return None

    except Exception as e:
        print(f"Error extracting recipe from {url}: {e}")
        return None

In [7]:

def looks_like_match(
    expected: str,
    actual: str,
) -> bool:
    expected = expected.lower()
    actual = actual.lower()

    return expected in actual or actual in expected


In [5]:
url = "https://www.thevegspace.co.uk/lentil-ragu/"
extracted = extract_recipe(url)

if extracted:
    print(f"Extracted recipe: {extracted['title']}")
else:
    print("Failed to extract recipe.")

Scraper parse failed for https://www.thevegspace.co.uk/lentil-ragu/: recipe-scrapers (15.11.0) exception: Website (The website 'thevegspace.co.uk' isn't currently supported by recipe-scrapers!
---
If you have time to help us out, please report this as a feature 
request on our bugtracker.) not supported.
Extracted recipe: Rich Lentil Ragù with red wine


In [6]:
extracted

{'title': 'Rich Lentil Ragù with red wine',
 'image': 'https://www.thevegspace.co.uk/wp-content/uploads/2022/07/lentil-ragu-2.jpg'}

In [8]:
looks_like_match(
    "Rich Lentil Ragù with Red Wine",
    extracted["title"],
)

True

In [21]:
from ddgs import DDGS


def search_recipe(recipe_name: str):
    query = f"{recipe_name} recipe"

    with DDGS() as ddgs:
        results = list(
            ddgs.text(
                query,
                max_results=5,
            )
        )

    return results

In [10]:
import asyncio
from pathlib import Path

import requests


def _download_image_sync(
    image_url: str,
    path: str,
):
    response = requests.get(
        image_url,
        timeout=20,
        allow_redirects=True,
    )
    response.raise_for_status()

    with open(path, "wb") as f:
        f.write(response.content)


async def download_image(
    image_url: str,
    recipe_id: str,
    output_dir: str = "data/images",
) -> str:
    Path(output_dir).mkdir(
        parents=True,
        exist_ok=True,
    )

    path = f"{output_dir}/{recipe_id}.jpg"

    await asyncio.to_thread(
        _download_image_sync,
        image_url,
        path,
    )

    return path

In [16]:
import asyncio
from pathlib import Path
from urllib.parse import urlparse

import requests


def _download_recipe_image_sync(
    image_url: str,
    path: str,
):
    base_headers = {
        "User-Agent": "Mozilla/5.0 RecipeBot/1.0",
    }

    response = requests.get(
        image_url,
        headers=base_headers,
        timeout=20,
        allow_redirects=True,
    )

    if response.status_code == 403:
        parsed = urlparse(image_url)
        retry_headers = {
            "User-Agent": (
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            ),
            "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
            "Referer": f"{parsed.scheme}://{parsed.netloc}/",
        }
        response = requests.get(
            image_url,
            headers=retry_headers,
            timeout=20,
            allow_redirects=True,
        )

    response.raise_for_status()

    with open(path, "wb") as f:
        f.write(response.content)


async def download_recipe_image(
    image_url: str,
    recipe_id: str,
    output_dir: str = "data/images",
) -> str:
    Path(output_dir).mkdir(
        parents=True,
        exist_ok=True,
    )

    path = f"{output_dir}/{recipe_id}.jpg"

    await asyncio.to_thread(
        _download_recipe_image_sync,
        image_url,
        path,
    )

    return path

In [17]:
path_image = await download_recipe_image(extracted["image"], "1")

In [18]:
path_image

'data/images/1.jpg'

In [26]:


async def process_recipe(
    recipe: RecipeRow,
    ) -> RecipeResult:

    # STEP 1: Try original URL
    try:
        status_code, _ = await fetch_html(
            recipe.source_url
        )

        if status_code == 200:

            extracted = extract_recipe(
                recipe.source_url
            )

            if extracted:

                if looks_like_match(
                    recipe.name,
                    extracted["title"],
                ):

                    image_path = await download_recipe_image(
                        extracted["image"],
                        recipe.id,
                    )

                    return RecipeResult(
                        id=recipe.id,
                        status="success",
                        final_url=recipe.source_url,
                        image_url=extracted["image"],
                        image_path=image_path,
                        notes=None,
                    )

    except Exception as e:
        print(e)

    # STEP 2: Search fallback
    print("Original URL failed, trying search fallback...")
    try:
        results = search_recipe(recipe.name)

        for result in results:

            url = result["href"]

            extracted = extract_recipe(url)

            if not extracted:
                print(f"Failed to extract recipe from {url}")
                continue

            if looks_like_match(
                recipe.name,
                extracted["title"],
            ):

                image_path = await download_recipe_image(
                    extracted["image"],
                    recipe.id,
                )

                return RecipeResult(
                    id=recipe.id,
                    status="recovered",
                    final_url=url,
                    image_url=extracted["image"],
                    image_path=image_path,
                    notes="Recovered via search",
                )

    except Exception as e:
        print(e)

    return RecipeResult(
        id=recipe.id,
        status="failed",
        final_url=None,
        image_url=None,
        image_path=None,
        notes="Could not recover recipe",
    )

In [28]:
import pandas as pd



async def main(path: str = "../data/vegan_recipes_database.csv"):

    df = pd.read_csv(
        path
    )

    rows = [
        RecipeRow(
            id=str(row["id"]),
            name=row["name"],
            source_url=row["source_url"],
            source_name=row["source_name"],
        )
        for _, row in df.iterrows()
    ]

    results = []

    for row in rows:

        print(
            f"Processing: {row.name}"
        )

        result = await process_recipe(row)

        results.append(result.__dict__)

    results_df = pd.DataFrame(results)

    results_df.to_csv(
        "../data/results.csv",
        index=False,
    )

    print("Done")



In [29]:
await main("../data/recipe_clean_errors.csv")

Processing: Rich Lentil Ragù with Red Wine
Scraper parse failed for https://www.thevegspace.co.uk/lentil-ragu/: recipe-scrapers (15.11.0) exception: Website (The website 'thevegspace.co.uk' isn't currently supported by recipe-scrapers!
---
If you have time to help us out, please report this as a feature 
request on our bugtracker.) not supported.
Processing: Creamy Chickpea and Lentil Curry
Scraper parse failed for https://www.thevegspace.co.uk/chickpea-and-lentil-curry/: recipe-scrapers (15.11.0) exception: Website (The website 'thevegspace.co.uk' isn't currently supported by recipe-scrapers!
---
If you have time to help us out, please report this as a feature 
request on our bugtracker.) not supported.
Processing: Vegan Sausage Casserole with Borlotti Beans
Scraper parse failed for https://www.thevegspace.co.uk/recipe-bangers-borlotti-bean-stew/: recipe-scrapers (15.11.0) exception: Website (The website 'thevegspace.co.uk' isn't currently supported by recipe-scrapers!
---
If you have

In [21]:
from uuid import uuid4
import pandas as pd


def build_cleaned_and_error_csvs(
    source_csv: str = "../data/vegan_recipes_database.csv",
    results_csv: str = "../data/results.csv",
    cleaned_out_csv: str = "../data/cleaned_recipes.csv",
    errors_out_csv: str = "../data/recipe_clean_errors.csv",
):
    source_df = pd.read_csv(source_csv, dtype={"id": str})
    results_df = pd.read_csv(results_csv, dtype={"id": str})

    merged = source_df.merge(
        results_df[["id", "status", "final_url", "image_path", "notes"]],
        on="id",
        how="left",
    )

    cleaned_mask = merged["status"].isin(["success", "recovered"])
    cleaned_df = merged.loc[cleaned_mask, source_df.columns.tolist()].copy()

    cleaned_df = cleaned_df.rename(columns={"id": "original_id"})
    cleaned_df.insert(0, "id", [str(uuid4()) for _ in range(len(cleaned_df))])
    cleaned_df["source_url"] = merged.loc[cleaned_mask, "final_url"].values
    cleaned_df["image_path"] = merged.loc[cleaned_mask, "image_path"].values

    errors_mask = merged["status"].eq("failed")
    errors_df = merged.loc[errors_mask, source_df.columns.tolist()].copy()
    errors_df["error_message"] = merged.loc[errors_mask, "notes"].fillna("Unknown error").values

    cleaned_df.to_csv(cleaned_out_csv, index=False)
    errors_df.to_csv(errors_out_csv, index=False)

    return cleaned_df, errors_df

In [ ]:
cleaned_df, errors_df = build_cleaned_and_error_csvs()

print(f"cleaned_recipes.csv rows: {len(cleaned_df)}")
print(f"recipe_clean_errors.csv rows: {len(errors_df)}")

cleaned_df.head(3), errors_df.head(3)

cleaned_recipes.csv rows: 49
recipe_clean_errors.csv rows: 71


(                                      id original_id  \
 2   eb50a4ee-d92f-4b2f-90fd-3874f74ffd94           3   
 10  bba2caa4-827a-4303-b68d-9f7d74a029d3          11   
 11  177cb113-460d-4686-b200-9dd504ad586e          12   
 
                              name    source_name  \
 2                    Vegan Hotpot  The Veg Space   
 10               Tuscan Bean Stew  The Veg Space   
 11  Chestnut Mushroom Bourguignon  The Veg Space   
 
                                            source_url  recipe_type  \
 2         https://www.youtube.com/watch?v=VK2DPab_oYM  Main Course   
 10  https://www.kitchentreaty.com/hearty-tuscan-be...  Main Course   
 11     https://www.cookingbites.com/tags/bourguignon/  Main Course   
 
                                           ingredients  \
 2   [{"item":"potatoes","amount":"600g, thinly sli...   
 10  [{"item":"olive oil","amount":"3 tbsp"},{"item...   
 11  [{"item":"olive oil","amount":"3 tbsp"},{"item...   
 
                                    

In [22]:
async def process_failed_recipes_to_dataframes(
    failed_df: pd.DataFrame,
    images_output_dir: str = "data/images",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Retry only failed recipes and keep everything in-memory.

    Returns:
        recovered_df: Rows that were recovered on retry
        still_failed_df: Rows that still failed, with detailed exception messages
    """
    recovered_rows = []
    still_failed_rows = []

    for _, row in failed_df.iterrows():
        row_dict = row.to_dict()
        recipe = RecipeRow(
            id=str(row_dict["id"]),
            name=row_dict["name"],
            source_url=row_dict["source_url"],
            source_name=row_dict["source_name"],
        )

        debug_errors = []
        original_error = row_dict.get("error_message")

        candidate_urls = [recipe.source_url]
        try:
            search_results = search_recipe(recipe.name)
            for result in search_results:
                href = result.get("href")
                if href and href not in candidate_urls:
                    candidate_urls.append(href)
        except Exception as e:
            debug_errors.append(f"search_exception: {type(e).__name__}: {e}")

        recovered = None

        for candidate_url in candidate_urls:
            try:
                extracted = extract_recipe(candidate_url)

                if not extracted:
                    debug_errors.append(
                        f"extract_none: {candidate_url}"
                    )
                    continue

                if not looks_like_match(recipe.name, extracted["title"]):
                    debug_errors.append(
                        f"title_mismatch: {candidate_url} -> '{extracted['title']}'"
                    )
                    continue

                try:
                    image_path = await download_image(
                        extracted["image"],
                        recipe.id,
                        output_dir=images_output_dir,
                    )
                except Exception as e:
                    debug_errors.append(
                        f"image_download_exception: {type(e).__name__}: {e}; url={extracted.get('image')}"
                    )
                    continue

                recovered = {
                    **row_dict,
                    "final_url": candidate_url,
                    "image_url": extracted.get("image"),
                    "image_path": image_path,
                    "retry_status": "recovered",
                    "error_message": None,
                }
                break

            except Exception as e:
                debug_errors.append(
                    f"candidate_exception: {type(e).__name__}: {e}; url={candidate_url}"
                )

        if recovered:
            recovered_rows.append(recovered)
        else:
            merged_error_parts = []
            if original_error and str(original_error).strip():
                merged_error_parts.append(f"original_error: {original_error}")
            if debug_errors:
                merged_error_parts.extend(debug_errors)
            if not merged_error_parts:
                merged_error_parts.append("No additional debug details")

            still_failed_rows.append(
                {
                    **row_dict,
                    "retry_status": "failed",
                    "error_message": " | ".join(merged_error_parts),
                }
            )

    recovered_df = pd.DataFrame(recovered_rows)
    still_failed_df = pd.DataFrame(still_failed_rows)
    return recovered_df, still_failed_df

In [23]:
errors_df = pd.read_csv("../data/recipe_clean_errors.csv", dtype={"id": str})

In [24]:
recovered_df, still_failed_df = await process_failed_recipes_to_dataframes(errors_df)

Scraper parse failed for https://www.thevegspace.co.uk/lentil-ragu/: recipe-scrapers (15.11.0) exception: Website (The website 'thevegspace.co.uk' isn't currently supported by recipe-scrapers!
---
If you have time to help us out, please report this as a feature 
request on our bugtracker.) not supported.
Scraper parse failed for https://www.thevegspace.co.uk/chickpea-and-lentil-curry/: recipe-scrapers (15.11.0) exception: Website (The website 'thevegspace.co.uk' isn't currently supported by recipe-scrapers!
---
If you have time to help us out, please report this as a feature 
request on our bugtracker.) not supported.
Scraper parse failed for https://www.thevegspace.co.uk/recipe-bangers-borlotti-bean-stew/: recipe-scrapers (15.11.0) exception: Website (The website 'thevegspace.co.uk' isn't currently supported by recipe-scrapers!
---
If you have time to help us out, please report this as a feature 
request on our bugtracker.) not supported.
Scraper parse failed for https://www.thevegspa

In [25]:
len(recovered_df), len(still_failed_df)

(0, 71)